In [15]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import io
import re
import time
import requests

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 35
PROJECT_ROOT: C:\Users\mjbou\governance-framework


## World Bank Carbon Pricing Dashboard Pipeline

**Source:** World Bank Carbon Pricing Dashboard
**Access:** Automated — month-stamped xlsx, latest auto-detected by date iteration
**Download instructions:** See `docs/instructions_data_maintenance.md` — WB_CARBON section

### Framework usage
| Indicator | Concept | Role |
|-----------|---------|------|
| Carbon pricing existence (national) | Environmental/climate governance | Primary tier 1 |
| Jurisdictional emissions coverage % | Environmental/climate governance | Primary tier 1 (design) |
| Carbon price (US$/tCO2e) | Environmental/climate governance | Primary tier 1 (design) |
| Carbon revenue (US$m, for revenue/GDP downstream) | Environmental/climate governance | Supporting (economic materiality) |

### Scope and method (national-only)
- **National instruments only.** Subnational schemes (US states, Canadian provinces, Chinese
  pilots, Mexican states, Japanese cities, etc.) are EXCLUDED — this scores sovereign-level
  governance. Classification is fail-safe: only jurisdictions resolving to a sovereign ISO3
  via an explicit dictionary are kept; anything else is excluded automatically.
- **EU ETS attributed to all member states** (EU27 + Iceland, Liechtenstein, Norway = the
  "EU27+" jurisdiction). This is a deliberate exception to national-only, because most EU
  members have no separate national scheme. The member list is a MANUAL UPDATE point.

### Honest limitations
- **Mixed time basis:** price and revenue are full panels; jurisdictional coverage is a
  CURRENT SNAPSHOT (the dashboard only provides current jurisdictional coverage, not history).
  Coverage is therefore cross-sectional and applied as the latest-known value.
- **Subnational excluded** — understates federal carbon-pricing economies (US, Canada, China).
- **Revenue understates free-allocation ETSs** (permits given free raise little revenue).
- **Within-country coverage uses MAX across instruments** (dashboard warns figures are gross
  and overlap; max is the conservative choice vs summing).
- True carbon-pricing *design quality* beyond these is out of scope.

### Interpreting absence (important for scoring)
A country absent from this panel must be treated as INFERRED, not VERIFIED, non-existence
of carbon pricing. Absence can mean genuinely no scheme, OR:
- the instrument falls outside the dashboard's scope (it tracks carbon taxes and ETSs only —
  not fuel excise with a carbon component, subsidy reform, or other implicit pricing);
- the country has only a SUBNATIONAL scheme (present in the source, filtered out here);
- the country has a scheme "under development / under consideration" (not counted in the
  existence flag, which requires Implemented status);
- reporting lag — a very new scheme may not be captured yet.
At the metric pass, an absent country may reasonably be scored as "no national carbon price,"
but should carry an inferred-absence flag so verified vs inferred absence stays distinguishable.
The World Bank dashboard is the most authoritative global tracker, so absence is decent
evidence against a MAJOR national tax/ETS — but it is not definitive.

In [17]:
# Download the latest WB Carbon Pricing Dashboard xlsx.
# File is month-stamped (data_{MM}_{YYYY}.xlsx); latest auto-detected by iterating months back.
# Server rate-limits intermittently, so retry with backoff. No hardcoded dates.
CARBON_BASE_PATHS = [
    "https://carbonpricingdashboard.worldbank.org/sites/default/files/carbon-pricing-dashboard-data/data_{m:02d}_{y}.xlsx",
    "https://carbonpricingdashboard.worldbank.org/sites/default/files/{y}-{m:02d}/data_{m:02d}_{y}.xlsx",
]

def fetch_latest_carbon_xlsx(months_back=24):
    """Find and download the most recent month-stamped dashboard xlsx. Returns (url, content, vintage_str)."""
    today = datetime.today()
    cmi = today.year * 12 + (today.month - 1)  # current month index for absolute-month arithmetic
    for back in range(months_back):
        total = cmi - back
        yr, mo = total // 12, total % 12 + 1
        for bp in CARBON_BASE_PATHS:
            url = bp.format(m=mo, y=yr)
            for attempt in range(2):
                try:
                    r = requests.get(url, headers=BROWSER_HEADERS, timeout=30)
                    if r.status_code == 200 and 'spreadsheet' in r.headers.get('Content-Type', '').lower():
                        return url, r.content, f"{yr}-{mo:02d}"
                    if r.status_code == 429:
                        time.sleep(3)  # rate-limited — back off and retry
                except requests.exceptions.RequestException:
                    time.sleep(2)
    return None, None, None

carbon_url, carbon_content, carbon_vintage = fetch_latest_carbon_xlsx()
if carbon_url is None:
    raise RuntimeError("Could not fetch WB Carbon dashboard xlsx — likely rate-limited. Wait and re-run.")

print(f"Fetched vintage {carbon_vintage}: {carbon_url}")

# Load the sheets we use: Gen Info (spine), Price, Revenue
xl = pd.ExcelFile(io.BytesIO(carbon_content), engine='openpyxl')
gen_info = xl.parse('Compliance_Gen Info', header=4)
print(f"Gen Info shape: {gen_info.shape}")
print(f"Columns sample: {list(gen_info.columns[:8])}")

Fetched vintage 2025-08: https://carbonpricingdashboard.worldbank.org/sites/default/files/carbon-pricing-dashboard-data/data_08_2025.xlsx
Gen Info shape: (131, 44)
Columns sample: ['Unique ID', 'Instrument name', 'Type', 'Status', 'Jurisdiction covered', 'Share of jurisdiction emissions covered', 'Price on 1 April', 2020]


In [19]:
# National jurisdiction name -> ISO3 mapping (explicit dict for transparency, no dependency).
# FAIL-SAFE: only jurisdictions in this dict are treated as national. Anything not listed
# (subnational schemes — US states, Canadian provinces, Chinese pilots, etc.) is EXCLUDED.
# MANUAL UPDATE: if a NEW SOVEREIGN country adopts carbon pricing, add it here. A diagnostic
# at the end of this cell prints any unmapped jurisdictions so new national entrants are caught.
NATIONAL_NAME_TO_ISO3 = {
    'Albania': 'ALB', 'Andorra': 'AND', 'Argentina': 'ARG', 'Australia': 'AUS',
    'Austria': 'AUT', 'Bahrain': 'BHR', 'Botswana': 'BWA', 'Brazil': 'BRA',
    'Brunei Darussalam': 'BRN', 'Canada': 'CAN', 'Chile': 'CHL', 'China': 'CHN',
    'Colombia': 'COL', 'Côte d’Ivoire': 'CIV', 'Denmark': 'DNK', 'Dominican Republic': 'DOM',
    'Estonia': 'EST', 'Finland': 'FIN', 'France': 'FRA', 'Germany': 'DEU',
    'Hungary': 'HUN', 'Iceland': 'ISL', 'India': 'IND', 'Indonesia': 'IDN',
    'Ireland': 'IRL', 'Israel': 'ISR', 'Japan': 'JPN', 'Kazakhstan': 'KAZ',
    'Kenya': 'KEN', 'Korea, Rep.': 'KOR', 'Latvia': 'LVA', 'Liechtenstein': 'LIE',
    'Luxembourg': 'LUX', 'Malaysia': 'MYS', 'Mauritania': 'MRT', 'Mexico': 'MEX',
    'Montenegro': 'MNE', 'Morocco': 'MAR', 'Netherlands': 'NLD', 'New Zealand': 'NZL',
    'Norway': 'NOR', 'Pakistan': 'PAK', 'Paraguay': 'PRY', 'Philippines': 'PHL',
    'Poland': 'POL', 'Portugal': 'PRT', 'Senegal': 'SEN', 'Singapore': 'SGP',
    'Slovenia': 'SVN', 'South Africa': 'ZAF', 'Spain': 'ESP', 'Sweden': 'SWE',
    'Switzerland': 'CHE', 'Taiwan, China': 'TWN', 'Thailand': 'THA', 'Türkiye': 'TUR',
    'Ukraine': 'UKR', 'United Kingdom': 'GBR', 'Uruguay': 'URY', 'Viet Nam': 'VNM',
}

# EU ETS jurisdiction label in the data is 'EU27+' = EU27 + Iceland, Liechtenstein, Norway (EEA).
# EU ETS instruments are attributed to ALL these members.
# MANUAL UPDATE: revise if EU/EEA ETS membership changes.
EU_ETS_JURISDICTION_LABEL = 'EU27+'
EU27_PLUS_ISO3 = [
    'AUT','BEL','BGR','HRV','CYP','CZE','DNK','EST','FIN','FRA','DEU','GRC','HUN',
    'IRL','ITA','LVA','LTU','LUX','MLT','NLD','POL','PRT','ROU','SVK','SVN','ESP','SWE',  # EU27
    'ISL','LIE','NOR',  # EEA states in EU ETS
]

def _clean_jur(jur):
    """Normalize a jurisdiction label: strip whitespace including non-breaking spaces (\xa0)."""
    if not isinstance(jur, str):
        return jur
    return jur.replace('\xa0', ' ').strip()

def jurisdiction_to_iso3_list(jur):
    """Map a 'Jurisdiction covered' label to a list of ISO3 codes.
    National name -> [iso3]; EU27+ -> all member states; subnational/unknown -> [] (excluded).
    Labels are whitespace-normalized first to avoid silent mismatches."""
    jur = _clean_jur(jur)
    if jur == EU_ETS_JURISDICTION_LABEL:
        return list(EU27_PLUS_ISO3)
    iso3 = NATIONAL_NAME_TO_ISO3.get(jur)
    return [iso3] if iso3 else []

# Diagnostic: list jurisdictions that are neither mapped national nor EU27+ — i.e. excluded.
# Helps catch a NEW NATIONAL entrant that should be added to the dict (vs expected subnationals).
all_juris = sorted({_clean_jur(j) for j in gen_info['Jurisdiction covered'].dropna().unique()})
unmapped = [j for j in all_juris
            if j != EU_ETS_JURISDICTION_LABEL and j not in NATIONAL_NAME_TO_ISO3]
print(f"National jurisdictions mapped: {len(NATIONAL_NAME_TO_ISO3)}")
print(f"Excluded (subnational/unknown) — {len(unmapped)} jurisdictions:")
print(unmapped)

National jurisdictions mapped: 60
Excluded (subnational/unknown) — 47 jurisdictions:
['Alberta', 'Baja California', 'Beijing', 'British Columbia', 'California', 'Catalonia', 'Chongqing', 'Colima', 'Colorado', 'Durango', 'Fujian', 'Guanajuato', 'Guangdong (except Shenzhen)', 'Hawaii', 'Hubei', 'Jalisco', 'Maryland', 'Massachusetts', 'Mexico City', 'Morelos', 'New Brunswick', 'New Jersey', 'New York State', 'Newfoundland and Labrador', 'Northwest Territories', 'Nova Scotia', 'Ontario', 'Oregon', 'Pennsylvania', 'Prince Edward Island', 'Quebec', 'Queretaro', 'RGGI', 'Saitama', 'Sakhalin', 'San Luis Potosí', 'Saskatchewan', 'Shanghai', 'Shenzhen', 'State of Mexico', 'Tamaulipas', 'Tianjin', 'Tokyo', 'Vermont', 'Washington', 'Yucatan', 'Zacatecas']


In [20]:
# COVERAGE (current snapshot). Parse jurisdictional coverage % from the Gen Info text field
# e.g. "73% of jurisdiction emissions, 0.0106% of global emissions" -> 73.0
# Map each instrument to country/countries (EU ETS -> all members), keep national only,
# aggregate within country using MAX (dashboard warns coverage is gross/overlapping; max is conservative).

def parse_jurisdiction_pct(text):
    """Extract the jurisdictional coverage percentage from the Gen Info coverage string."""
    if not isinstance(text, str):
        return None
    # Match the first "<num>% of jurisdiction emissions"
    m = re.search(r'([\d.]+)\s*%\s*of\s*jurisdiction', text, flags=re.IGNORECASE)
    return float(m.group(1)) if m else None

# Work on a copy with normalized jurisdiction + parsed coverage
gi = gen_info.copy()
gi['jur_clean'] = gi['Jurisdiction covered'].map(_clean_jur)
gi['cov_pct'] = gi['Share of jurisdiction emissions covered'].map(parse_jurisdiction_pct)

# Expand each instrument to its country list (national only; EU ETS -> members; else dropped)
cov_rows = []
for _, row in gi.iterrows():
    for iso3 in jurisdiction_to_iso3_list(row['Jurisdiction covered']):
        if row['cov_pct'] is not None:
            cov_rows.append({'country_code': iso3, 'wb_carbon_coverage_pct': row['cov_pct']})

cov_df = pd.DataFrame(cov_rows)
# Aggregate within country: MAX coverage across the country's instruments (conservative re overlap)
coverage = cov_df.groupby('country_code', as_index=False)['wb_carbon_coverage_pct'].max()

print(f"Coverage: {coverage.shape[0]} countries with a parsed jurisdictional coverage %")
print(coverage.sort_values('wb_carbon_coverage_pct', ascending=False).head(10).to_string(index=False))

Coverage: 71 countries with a parsed jurisdictional coverage %
country_code  wb_carbon_coverage_pct
         AND                    95.0
         ZAF                    82.0
         JPN                    80.0
         KOR                    79.0
         ISR                    78.0
         ALB                    73.0
         LUX                    72.0
         LIE                    72.0
         SGP                    71.0
         NOR                    65.0


In [21]:
# PRICE panel (US$/tCO2e). Price sheet is instrument-level wide by year.
# Join to Gen Info on Unique ID to get each instrument's jurisdiction, map to country
# (EU ETS -> members), keep national only, aggregate simple MEAN within country-year.

# Load Price sheet — header on row index 1 (0-based)
price_raw = xl.parse('Compliance_Price', header=1)

# Identify year columns (numeric/float-like headers) vs metadata columns
price_year_cols = [c for c in price_raw.columns if isinstance(c, (int, float))]
price_meta = price_raw[['Unique ID']].copy()

# Melt year columns to long form
price_long = price_raw.melt(
    id_vars=['Unique ID'],
    value_vars=price_year_cols,
    var_name='year',
    value_name='price',
)
price_long['year'] = price_long['year'].astype(int)
price_long = price_long.dropna(subset=['price'])

# Map each Unique ID to its jurisdiction via Gen Info, then to country list
uid_to_jur = dict(zip(gen_info['Unique ID'], gen_info['Jurisdiction covered']))
def uid_to_countries(uid):
    return jurisdiction_to_iso3_list(uid_to_jur.get(uid))

# Expand each instrument-year to its country list (national only; EU ETS -> members)
price_rows = []
for _, r in price_long.iterrows():
    for iso3 in uid_to_countries(r['Unique ID']):
        price_rows.append({'country_code': iso3, 'year': r['year'], 'price': r['price']})

price_df = pd.DataFrame(price_rows)
# Simple mean price across a country's national instruments in each year
price_panel = price_df.groupby(['country_code', 'year'], as_index=False)['price'].mean()
price_panel = price_panel.rename(columns={'price': 'wb_carbon_price_usd'})

print(f"Price panel: {price_panel.shape[0]} country-years, {price_panel['country_code'].nunique()} countries")
print(f"Year range: {price_panel['year'].min()} — {price_panel['year'].max()}")
print("Sample — highest recent prices:")
recent = price_panel[price_panel['year'] == price_panel['year'].max()]
print(recent.sort_values('wb_carbon_price_usd', ascending=False).head(8).to_string(index=False))

Price panel: 848 country-years, 51 countries
Year range: 1994 — 2025
Sample — highest recent prices:
country_code  year  wb_carbon_price_usd
         URY  2025           158.764742
         SWE  2025           107.497411
         LIE  2025           103.204561
         NOR  2025           102.145305
         CHE  2025           100.394287
         DNK  2025            89.402452
         NLD  2025            82.598322
         PRT  2025            71.535228


In [24]:
# REVENUE panel (US$ millions). Revenue is EXTENSIVE (a total), unlike price/coverage which are
# intensive. The EU ETS revenue is reported as a single BLOC figure, not per-member, so it
# CANNOT be attributed to members without double-counting (~30x overstatement). Therefore revenue
# uses national-only mapping with NO EU expansion: each country gets only its OWN national scheme
# revenue. EU members without a separate national scheme will have no carbon-revenue value here.
# This understates EU members' true revenue but never overstates. Documented limitation.
# Revenue stored raw; revenue/GDP computed downstream at metric pass using WDI GDP.

revenue_raw = xl.parse('Compliance_Revenue', header=1)
rev_year_cols = [c for c in revenue_raw.columns if isinstance(c, (int, float))]
rev_id_col = 'Unique ID' if 'Unique ID' in revenue_raw.columns else 'Instrument name'

rev_long = revenue_raw.melt(
    id_vars=[rev_id_col],
    value_vars=rev_year_cols,
    var_name='year',
    value_name='revenue',
)
rev_long['year'] = rev_long['year'].astype(int)
rev_long = rev_long.dropna(subset=['revenue'])

# Map instrument -> jurisdiction
if rev_id_col == 'Unique ID':
    id_to_jur = dict(zip(gen_info['Unique ID'], gen_info['Jurisdiction covered']))
else:
    id_to_jur = dict(zip(gen_info['Instrument name'], gen_info['Jurisdiction covered']))

def revenue_country(jur):
    """National-only mapping for revenue: maps a national jurisdiction to its ISO3.
    Returns None for subnational AND for the EU27+ bloc (revenue not attributable per-member)."""
    jur_clean = _clean_jur(jur)
    if jur_clean == EU_ETS_JURISDICTION_LABEL:
        return None  # bloc revenue — do NOT fan out to members (would multiply ~30x)
    return NATIONAL_NAME_TO_ISO3.get(jur_clean)

rev_rows = []
for _, r in rev_long.iterrows():
    iso3 = revenue_country(id_to_jur.get(r[rev_id_col]))
    if iso3:
        rev_rows.append({'country_code': iso3, 'year': r['year'], 'revenue': r['revenue']})

rev_df = pd.DataFrame(rev_rows)
revenue_panel = rev_df.groupby(['country_code', 'year'], as_index=False)['revenue'].sum()
revenue_panel = revenue_panel.rename(columns={'revenue': 'wb_carbon_revenue_usd_m'})

print(f"Revenue panel: {revenue_panel.shape[0]} country-years, {revenue_panel['country_code'].nunique()} countries")
print("Sample — highest recent revenue (US$m):")
recent_rev = revenue_panel[revenue_panel['year'] == revenue_panel['year'].max()]
print(recent_rev.sort_values('wb_carbon_revenue_usd_m', ascending=False).head(8).to_string(index=False))

Revenue panel: 489 country-years, 39 countries
Sample — highest recent revenue (US$m):
country_code  year  wb_carbon_revenue_usd_m
         DEU  2024             13933.168740
         CAN  2024              8858.771097
         FRA  2024              7843.850000
         GBR  2024              4121.536262
         SWE  2024              2306.382062
         NOR  2024              1605.429133
         CHE  2024              1475.834221
         JPN  2024              1451.883105


In [25]:
# ASSEMBLE final panel (skeleton = UNION of all country-years with ANY carbon information,
# so no country with coverage/existence is silently dropped for lacking price/revenue).
# Existence and coverage are current facts (no native year); per decision, countries that have
# a scheme but NO price/revenue series get a single CURRENT-YEAR row (latest year in the data),
# rather than broadcasting unverified history.

# --- Existence: implemented national instruments (EU ETS expanded to members; intensive) ---
gi2 = gen_info.copy()
gi2['status_clean'] = gi2['Status'].map(_clean_jur)
exist_countries = set()
for _, row in gi2.iterrows():
    if str(row['status_clean']).strip().lower() == 'implemented':
        for iso3 in jurisdiction_to_iso3_list(row['Jurisdiction covered']):
            exist_countries.add(iso3)
print(f"Countries with an implemented national carbon-pricing instrument: {len(exist_countries)}")

# --- Current year derived from the data (latest year in the price panel) — no hardcoding ---
current_year = int(price_panel['year'].max())

# --- Base from price + revenue country-years ---
base = pd.merge(price_panel, revenue_panel, on=['country_code', 'year'], how='outer')

# --- Ensure every existence country and every coverage country has at least one row ---
# Countries already in base keep their full year series; those absent get a single current-year row.
present_countries = set(base['country_code'].unique())
extra_countries = (exist_countries | set(coverage['country_code'])) - present_countries
if extra_countries:
    extra_rows = pd.DataFrame({'country_code': sorted(extra_countries), 'year': current_year})
    base = pd.concat([base, extra_rows], ignore_index=True)
print(f"Added single-current-year rows for {len(extra_countries)} scheme/coverage-only countries: {sorted(extra_countries)}")

# --- Broadcast coverage snapshot (static current value) onto all rows, flagged as snapshot ---
base = base.merge(coverage, on='country_code', how='left')
base['wb_carbon_coverage_is_snapshot'] = base['wb_carbon_coverage_pct'].notna().astype(int)

# --- Existence flag per row ---
base['wb_carbon_pricing_exists'] = base['country_code'].isin(exist_countries).astype(int)

# Filter to framework start year, tidy, reorder
panel = base[base['year'] >= FRAMEWORK_START_YEAR].copy()
panel = panel.sort_values(['country_code', 'year']).reset_index(drop=True)
panel = panel[['country_code', 'year', 'wb_carbon_pricing_exists',
               'wb_carbon_price_usd', 'wb_carbon_revenue_usd_m',
               'wb_carbon_coverage_pct', 'wb_carbon_coverage_is_snapshot']]

print(f"\nFinal panel: {panel.shape}")
print(f"Years: {panel['year'].min()} — {panel['year'].max()}")
print(f"Countries: {panel['country_code'].nunique()}")
print(f"\n⚠️ COVERAGE FLAG: {panel['country_code'].nunique()} countries — materially below ~150.")
print("   Expected: national carbon pricing is genuinely concentrated in ~45-50 countries.")
print(f"\nMissing values (%):")
mp = (panel.isnull().sum() / len(panel) * 100).round(1)
print(mp[mp > 0].sort_values(ascending=False))

Countries with an implemented national carbon-pricing instrument: 53
Added single-current-year rows for 19 scheme/coverage-only countries: ['BHR', 'BRA', 'BRN', 'BWA', 'CIV', 'DOM', 'IND', 'KEN', 'MAR', 'MRT', 'MYS', 'PAK', 'PHL', 'PRY', 'SEN', 'THA', 'TUR', 'TWN', 'VNM']

Final panel: (928, 7)
Years: 1990 — 2025
Countries: 71

⚠️ COVERAGE FLAG: 71 countries — materially below ~150.
   Expected: national carbon pricing is genuinely concentrated in ~45-50 countries.

Missing values (%):
wb_carbon_revenue_usd_m    47.3
wb_carbon_price_usd         8.6
wb_carbon_coverage_pct      1.9
dtype: float64


In [27]:
# Derive data currency from the detected vintage — no hardcoding
data_as_of = carbon_vintage  # YYYY-MM of the dashboard file

# Save to processed
output_path = os.path.join(PROCESSED_DIR, "wb_carbon_clean.csv")
panel.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {panel.shape}")

n_countries = panel['country_code'].nunique()

# Update download log — coverage flagged in notes per the thin-coverage decision
update_entry(
    "WB_CARBON",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=data_as_of,
    local_filename="wb_carbon_clean.csv",
    latest_available_version=carbon_vintage,
    notes=(f"National carbon pricing: existence flag, price (US$/tCO2e), revenue (US$m), and "
           f"jurisdictional emissions coverage % (current SNAPSHOT, flagged). "
           f"National-only; subnational excluded. EU ETS expanded to members for intensive measures "
           f"(price, coverage, existence) but NOT revenue (bloc total — would ~30x overstate). "
           f"Within-country coverage = max across instruments. Price/revenue are panels; coverage is snapshot. "
           f"⚠️ COVERAGE: {n_countries} countries — materially below ~150 (carbon pricing is genuinely "
           f"concentrated; non-adopters simply absent, not a data defect). "
           f"Auto-detects latest month-stamped xlsx. Revenue/GDP to be computed at metric pass.")
)
print_entry("WB_CARBON")

Written: C:\Users\mjbou\governance-framework\data\processed\wb_carbon_clean.csv
Shape: (928, 7)
[download_log] Updated entry for WB_CARBON
  source_id: WB_CARBON
  last_attempted_date: 2026-06-12
  last_successful_download_date: 2026-06-17
  data_as_of_date: 2025-08
  local_filename: wb_carbon_clean.csv
  latest_available_version: 2025-08
  no_update_reason: nan
  notes: National carbon pricing: existence flag, price (US$/tCO2e), revenue (US$m), and jurisdictional emissions coverage % (current SNAPSHOT, flagged). National-only; subnational excluded. EU ETS expanded to members for intensive measures (price, coverage, existence) but NOT revenue (bloc total — would ~30x overstate). Within-country coverage = max across instruments. Price/revenue are panels; coverage is snapshot. ⚠️ COVERAGE: 71 countries — materially below ~150 (carbon pricing is genuinely concentrated; non-adopters simply absent, not a data defect). Auto-detects latest month-stamped xlsx. Revenue/GDP to be computed at met